<a href="https://colab.research.google.com/github/tranvananhanhanh/AI_Agent/blob/main/simple_ai_agent.py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ==========================================
# 1. Install library
# ==========================================
!pip install litellm


# ==========================================
# 2. Setup API key from Colab Secret
# ==========================================
import os
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key


# ==========================================
# 3. Imports
# ==========================================
import json
import re
from litellm import completion
from typing import List, Dict


# ==========================================
# 4. Tools
# ==========================================
def list_files():
    """List files in current directory"""
    return os.listdir(".")


def read_file(file_name):
    """Read contents of a file"""
    try:
        with open(file_name, "r") as f:
            return f.read()
    except Exception as e:
        return str(e)


def terminate(message):
    """Terminate agent and return message"""
    return message


# ==========================================
# 5. Agent Rules (System Prompt)
# ==========================================
agent_rules = """
You are an AI agent that can perform tasks using tools.

Available tools:

1. list_files()
   Returns a list of files in the current directory.

2. read_file(file_name: str)
   Reads the contents of a file.

3. terminate(message: str)
   Ends the task and returns a message to the user.

You MUST always respond with JSON in this format:

{
 "tool_name": "...",
 "args": {...}
}

Do not output anything else.
"""


# ==========================================
# 6. LLM Call
# ==========================================
def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""

    response = completion(
        model="groq/llama-3.1-8b-instant",
        messages=messages,
        max_tokens=1024
    )

    return response.choices[0].message.content


# ==========================================
# 7. Parse LLM Response
# ==========================================
def parse_action(response):

    try:
        json_text = re.search(r'\{.*\}', response, re.DOTALL).group()
        action = json.loads(json_text)
        return action
    except:
        return {
            "tool_name": "terminate",
            "args": {"message": "Failed to parse model response"}
        }


# ==========================================
# 8. Execute Tool
# ==========================================
def execute_action(action):

    tool = action["tool_name"]
    args = action["args"]

    if tool == "list_files":
        return list_files()

    elif tool == "read_file":
        return read_file(args.get("file_name"))

    elif tool == "terminate":
        return terminate(args.get("message"))

    else:
        return "Unknown tool"


# ==========================================
# 9. Agent Loop
# ==========================================
def run_agent(user_input, max_iterations=10):

    memory = [
        {"role": "system", "content": agent_rules},
        {"role": "user", "content": user_input}
    ]

    for step in range(max_iterations):

        print(f"\n--- Agent Step {step+1} ---")

        response = generate_response(memory)

        print("Model response:")
        print(response)

        action = parse_action(response)

        print("\nParsed Action:")
        print(action)

        result = execute_action(action)

        print("\nTool Result:")
        print(result)

        if action["tool_name"] == "terminate":
            return result

        memory.append({"role": "assistant", "content": response})
        memory.append({"role": "user", "content": json.dumps(result)})

    return "Max iterations reached"


# ==========================================
# 10. Run Agent
# ==========================================
question = "What files are in this directory?"

result = run_agent(question)

print("\n==========================")
print("FINAL RESULT:")
print(result)


--- Agent Step 1 ---
Model response:
{
  "tool_name": "list_files",
  "args": {}
}

Parsed Action:
{'tool_name': 'list_files', 'args': {}}

Tool Result:
['.config', 'sum_of_a_and_b.py', 'sample_data']

--- Agent Step 2 ---
Model response:
{
  "tool_name": "list_files",
  "args": {}
}

Parsed Action:
{'tool_name': 'list_files', 'args': {}}

Tool Result:
['.config', 'sum_of_a_and_b.py', 'sample_data']

--- Agent Step 3 ---
Model response:
{
  "tool_name": "read_file",
  "args": {
    "file_name": "sum_of_a_and_b.py"
  }
}

Parsed Action:
{'tool_name': 'read_file', 'args': {'file_name': 'sum_of_a_and_b.py'}}

Tool Result:

"""
Module for arithmetic operations.

This module provides a function to calculate the sum of two numbers.
"""


def add_numbers(a: int or float, b: int or float) -> float:
    """
    Calculate the sum of two numbers.

    Args:
        a (int or float): The first number.
        b (int or float): The second number.

    Returns:
        float: The sum of a and b.

 